# Experiment 5: Subword Tokenization and POS Tagging

## 5.1 BPE Tokenization

In [ ]:
# a. Using pretrained model (GPT-2 uses BPE)
from transformers import GPT2Tokenizer
tokenizer_bpe_pre = GPT2Tokenizer.from_pretrained('gpt2')

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    sample_text = f.read().strip().split('\n')[0] # Using first line as sample

print('--- BPE Pretrained ---')
encoded_bpe = tokenizer_bpe_pre.encode(sample_text)
tokens_bpe = tokenizer_bpe_pre.convert_ids_to_tokens(encoded_bpe)
for t, i in zip(tokens_bpe[:20], encoded_bpe[:20]):
    print(f'{t}: {i}')

In [ ]:
# b. Without pretrained model (Train from scratch)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer_bpe_scratch = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe_scratch.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=1000, special_tokens=['[UNK]'])
tokenizer_bpe_scratch.train(['input_sub_word_data.txt'], trainer)

print('\n--- BPE From Scratch ---')
encoded_scratch = tokenizer_bpe_scratch.encode(sample_text)
for t, i in zip(encoded_scratch.tokens[:20], encoded_scratch.ids[:20]):
    print(f'{t}: {i}')

## 5.2 SentencePiece Tokenization

In [ ]:
# a. Using pretrained model (T5 uses SentencePiece)
from transformers import T5Tokenizer
tokenizer_sp_pre = T5Tokenizer.from_pretrained('t5-small', legacy=False)

print('\n--- SentencePiece Pretrained ---')
encoded_sp = tokenizer_sp_pre.encode(sample_text)
tokens_sp = tokenizer_sp_pre.convert_ids_to_tokens(encoded_sp)
for t, i in zip(tokens_sp[:20], encoded_sp[:20]):
    print(f'{t}: {i}')

In [ ]:
# b. Without pretrained model (Train from scratch)
import sentencepiece as spm
spm.SentencePieceTrainer.train(input='input_sub_word_data.txt', model_prefix='m_sp', vocab_size=500, user_defined_symbols=['<unk>'])
sp = spm.SentencePieceProcessor(model_file='m_sp.model')

print('\n--- SentencePiece From Scratch ---')
encoded_sp_scratch = sp.encode_as_ids(sample_text)
tokens_sp_scratch = sp.encode_as_pieces(sample_text)
for t, i in zip(tokens_sp_scratch[:20], encoded_sp_scratch[:20]):
    print(f'{t}: {i}')

## 5.3 POS Tagging (Short Sentence)

In [ ]:
sentence = 'The young student is reading an interesting book in the library.'

# a. Using spaCy
import spacy
nlp = spacy.load('en_core_web_sm')
doc = nlp(sentence)

print('\n--- POS Tagging (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token in doc:
    print(f'{token.text:<15} | {token.pos_:<10} | {spacy.explain(token.pos_)}')

# b. Using NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

tokens_nltk = nltk.word_tokenize(sentence)
pos_tags_nltk = nltk.pos_tag(tokens_nltk)

print('\n--- POS Tagging (NLTK) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token, tag in pos_tags_nltk:
    print(f'{token:<15} | {tag:<10} | {tag}')

## 5.4 POS Tagging with Frequency (Large File)

In [ ]:
from collections import Counter

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    large_text = f.read()

# Process first 5000 chars for demonstration
doc_large = nlp(large_text[:5000])

# Count frequencies of (Token, POS)
pos_freq = Counter([(token.text, token.pos_, spacy.explain(token.pos_)) for token in doc_large if not token.is_space])

print('\n--- POS Tagging with Frequency (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Frequency":<10} | {"Description"}')
print('-'*65)
for (token, pos, desc), freq in pos_freq.most_common(20):
    print(f'{token:<15} | {pos:<10} | {freq:<10} | {desc}')